In [0]:
df=spark.table("customer_pipeline.customer_data.silver_customers")
display(df)


customer_id,customer_name,city,email,age,salary,email_status
1003,Priya,Hyderabad,priya@gmail.com,24.0,46000.0,Valid
1015,Ajay,Bangalore,ajay@gmail.com,29.0,53000.0,Valid
1022,Swathi,Bangalore,swathi@gmail.com,24.0,0.0,Valid
1025,Asha,Chennai,asha@gmail.com,29.0,54000.0,Valid
1002,Arun,Chennai,arun@gmail.com,28.0,51000.0,Valid
1007,Vijay,Bangalore,vijay@gmail.com,32.0,58000.0,Valid
1008,Anita,Delhi,anita@gmail.com,26.0,45000.0,Valid
1011,Manoj,Bangalore,manoj@gmail.com,35.0,70000.0,Valid
1024,Varun,Delhi,varun@gmail.com,27.0,50000.0,Valid
1001,Rahul,Bangalore,rahul@gmail.com,25.0,42000.0,Valid


In [0]:
# Step 3: Calculate total customers
print("Total_customers:",df.count())

Total_customers: 22


In [0]:
# Step 4: Find total customers in each city
from pyspark.sql.functions import count
city_summary=df.groupBy("city")\
    .agg(count("customer_id").alias("total_customers"))
display(city_summary)

city,total_customers
Hyderabad,2
Bangalore,8
Chennai,4
Delhi,3
unknown,1
Pune,2
Mumbai,2


In [0]:
# Step 5: Calculate average salary by city
from pyspark.sql.functions import avg,round

salary_summary=df.groupBy("city")\
    .agg(round(avg("salary"),2).alias("avg_salary"))
display(salary_summary)

city,avg_salary
Hyderabad,42500.0
Bangalore,46500.0
Chennai,56250.0
Delhi,44333.33
unknown,47000.0
Pune,58000.0
Mumbai,55500.0


In [0]:
# Step 6: Find customers earning more than 50,000
high_salary=df.filter("salary>50000").show()

+-----------+-------------+---------+-----------------+----+-------+------------+
|customer_id|customer_name|     city|            email| age| salary|email_status|
+-----------+-------------+---------+-----------------+----+-------+------------+
|       1015|         Ajay|Bangalore|   ajay@gmail.com|29.0|53000.0|       Valid|
|       1025|         Asha|  Chennai|   asha@gmail.com|29.0|54000.0|       Valid|
|       1002|         Arun|  Chennai|   arun@gmail.com|28.0|51000.0|       Valid|
|       1007|        Vijay|Bangalore|  vijay@gmail.com|32.0|58000.0|       Valid|
|       1011|        Manoj|Bangalore|  manoj@gmail.com|35.0|70000.0|       Valid|
|       1019|       Sanjay|Bangalore| sanjay@gmail.com| 0.0|59000.0|       Valid|
|       1006|        Meena|     Pune|  meena@gmail.com|29.0|55000.0|       Valid|
|       1009|         Ravi|  Chennai|   ravi@gmail.com|30.0|52000.0|       Valid|
|       1005|        Kiran|   Mumbai|  kiran@gmail.com|31.0|62000.0|       Valid|
|       1013|   

In [0]:
# Step 7: Find the city with the highest number of customers
top_city = city_summary.orderBy(
    "total_customers", ascending=False
)
display(top_city)

city,total_customers
Bangalore,8
Chennai,4
Delhi,3
Mumbai,2
Hyderabad,2
Pune,2
unknown,1


In [0]:
# Step 8: Create Gold summary table
city_summary.write.format("delta")\
    .mode("overwrite")\
        .saveAsTable("customer_pipeline.customer_data.gold_customer_summary")

        

In [0]:
%sql
select * from customer_pipeline.customer_data.gold_customer_summary

city,total_customers
Delhi,3
Chennai,4
Hyderabad,2
Bangalore,8
Pune,2
Mumbai,2
unknown,1
